<a href="https://colab.research.google.com/github/jaidvedant/AML/blob/main/AML_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from mlxtend.feature_selection import SequentialFeatureSelector as SFS


# 1. Load Dataset
df = sns.load_dataset('titanic')
# Select useful columns
df = df[['survived','pclass','sex','age','fare','sibsp','parch','embarked']]
print("Dataset Head:")
print(df.head())

# 2. Preprocessing
# Handle missing values
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Encode categorical columns
le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])
df['embarked'] = le.fit_transform(df['embarked'])

# Features and target
X = df.drop('survived', axis=1)
y = df['survived']

# Feature scaling
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 3. Logistic Regression Model
model = LogisticRegression()


# 4. Sequential Forward Selection
sfs = SFS(model,
          k_features=4,
          forward=True,
          floating=False,
          scoring='accuracy',
          cv=5)

sfs.fit(X_train, y_train)

forward_features = list(sfs.k_feature_names_)
print("\nTop Features using Forward Selection:")
print(forward_features)

model.fit(X_train[forward_features], y_train)
pred1 = model.predict(X_test[forward_features])
acc1 = accuracy_score(y_test, pred1)
print("Forward Selection Accuracy:", acc1)

# 5. Sequential Backward Elimination
sbe = SFS(model,
          k_features=4,
          forward=False,
          floating=False,
          scoring='accuracy',
          cv=5)

sbe.fit(X_train, y_train)

backward_features = list(sbe.k_feature_names_)
print("\nTop Features using Backward Elimination:")
print(backward_features)

model.fit(X_train[backward_features], y_train)
pred2 = model.predict(X_test[backward_features])
acc2 = accuracy_score(y_test, pred2)
print("Backward Elimination Accuracy:", acc2)

# 6. Comparison
print("\nPerformance Comparison:")
print("SFS Accuracy:", acc1)
print("SBE Accuracy:", acc2)
if acc1 > acc2:
    print("Sequential Forward Selection performed better.")
elif acc2 > acc1:
    print("Sequential Backward Elimination performed better.")
else:
    print("Both methods performed equally.")

Dataset Head:
   survived  pclass     sex   age     fare  sibsp  parch embarked
0         0       3    male  22.0   7.2500      1      0        S
1         1       1  female  38.0  71.2833      1      0        C
2         1       3  female  26.0   7.9250      0      0        S
3         1       1  female  35.0  53.1000      1      0        S
4         0       3    male  35.0   8.0500      0      0        S

Top Features using Forward Selection:
['pclass', 'sex', 'sibsp', 'parch']
Forward Selection Accuracy: 0.7932960893854749

Top Features using Backward Elimination:
['pclass', 'sex', 'sibsp', 'parch']
Backward Elimination Accuracy: 0.7932960893854749

Performance Comparison:
SFS Accuracy: 0.7932960893854749
SBE Accuracy: 0.7932960893854749
Both methods performed equally.
